# Capítulo 1 — Órdenes de magnitud y estimaciones de Fermi

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Cómo se descompone un problema imposible en cinco problemas fáciles?

Diagrama de la anatomía de una estimación de Fermi, con el ejemplo de la
energía de una tormenta. Responde: ¿qué se hace exactamente cuando alguien
dice «descomponer el problema»?

Ejecutar:  python fig_anatomia.py

*(script original: `codigo/fig_anatomia.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

from estilo_libro import C, caja, flecha, lienzo, save, use_style  # noqa: E402

use_style()
fig, ax = lienzo(ancho=9.2, alto=5.4, xlim=(0, 12), ylim=(0, 7))
ax.set_aspect("auto")

# --- Pregunta imposible ---------------------------------------------------
caja(ax, 6, 6.3, 7.4, 0.9,
     "¿Cuánta energía libera una tormenta de verano?",
     color=C.ink, relleno="white", fontsize=10.5)

# --- Los factores ---------------------------------------------------------
FACTORES = [
    ("Área\nde la célula", "$A \\sim 10\\times10$ km\n$=10^{8}$ m$^2$", C.blue),
    ("Lluvia\ncaída", "$h \\sim 20$ mm\n$=2\\times10^{-2}$ m", C.blue),
    ("Densidad\ndel agua", "$\\rho = 10^{3}$\nkg/m$^3$", C.green),
    ("Calor latente\nde condensación", "$L = 2{,}3\\times10^{6}$\nJ/kg", C.green),
]
x0, dx = 1.7, 2.9
for i, (titulo, valor, color) in enumerate(FACTORES):
    x = x0 + i * dx
    caja(ax, x, 4.3, 2.5, 1.5, f"{titulo}\n\n{valor}", color=color, fontsize=8.6)
    flecha(ax, (6, 5.85), (x, 5.05), color=C.grey, rad=0.0, lw=1.0)

ax.text(0.35, 4.3, "1. Descomponer", fontsize=9, color=C.ink,
        rotation=90, va="center", ha="center", weight="bold")

# --- Combinación ----------------------------------------------------------
caja(ax, 6, 2.5, 8.4, 1.0,
     "$E \\;=\\; A \\cdot h \\cdot \\rho \\cdot L \\;=\\; "
     "10^{8}\\cdot 2\\!\\times\\!10^{-2}\\cdot 10^{3}\\cdot 2{,}3\\!\\times\\!10^{6}"
     "\\;\\approx\\; 5\\times10^{15}\\ \\mathrm{J}$",
     color=C.red, relleno="#fdf3f2", fontsize=10.5)
for i in range(4):
    flecha(ax, (x0 + i * dx, 3.55), (6, 3.05), color=C.grey, lw=1.0)
ax.text(0.35, 2.5, "2. Multiplicar", fontsize=9, color=C.ink,
        rotation=90, va="center", ha="center", weight="bold")

# --- Contraste ------------------------------------------------------------
caja(ax, 3.1, 0.8, 4.4, 1.0,
     "¿Contra qué lo comparo?\n70 bombas de Hiroshima", color=C.ochre, fontsize=9)
caja(ax, 8.6, 0.8, 4.4, 1.0,
     "¿Dónde está mi error?\nen $h$ y en $A$, no en $\\rho$ ni en $L$",
     color=C.purple, fontsize=9)
flecha(ax, (6, 1.95), (3.1, 1.35), color=C.grey, lw=1.0)
flecha(ax, (6, 1.95), (8.6, 1.35), color=C.grey, lw=1.0)
ax.text(0.35, 0.8, "3. Criticar", fontsize=9, color=C.ink,
        rotation=90, va="center", ha="center", weight="bold")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Son honestos tus intervalos de confianza?

Simula tres estimadores con el mismo acierto medio pero distinta honestidad
al declarar su incertidumbre, y dibuja la curva de calibración: qué fracción
de los intervalos del x % contiene realmente el valor verdadero.

La figura responde: ¿cómo se detecta el exceso de confianza sin hacer
psicología, sólo contando?

Ejecutar:  python fig_calibracion.py

*(script original: `codigo/fig_calibracion.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
aleatorio = rng(2024)

N_PREGUNTAS = 5000
SIGMA_REAL = 0.55        # dispersión real del error, en décadas

# Tres personas: declaran una sigma distinta de la que realmente tienen
PERSONAS = {
    "Exceso de confianza\n(declara $\\sigma/3$)": SIGMA_REAL / 3,
    "Calibrado\n(declara su $\\sigma$)": SIGMA_REAL,
    "Exceso de prudencia\n(declara $2\\sigma$)": SIGMA_REAL * 2,
}
COLORES = [C.red, C.green, C.blue]

errores = aleatorio.normal(0.0, SIGMA_REAL, N_PREGUNTAS)
niveles = np.linspace(0.05, 0.99, 40)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.8, 4.1),
                               gridspec_kw={"width_ratios": [1, 1]})

# --- Curva de calibración -------------------------------------------------
ax1.plot([0, 1], [0, 1], "--", color=C.grey, lw=1.2,
         label="honestidad perfecta")
for (etiqueta, sigma_declarada), color in zip(PERSONAS.items(), COLORES):
    cobertura = [
        np.mean(np.abs(errores) <= norm.ppf(0.5 + p / 2) * sigma_declarada)
        for p in niveles
    ]
    ax1.plot(niveles, cobertura, color=color, lw=2.0, label=etiqueta)
ax1.set_xlabel("nivel declarado del intervalo")
ax1.set_ylabel("fracción que contiene el valor real")
ax1.set_title("Curva de calibración")
ax1.legend(fontsize=8, loc="lower right")
ax1.set_xlim(0, 1), ax1.set_ylim(0, 1)

# --- Lo que se mide en la práctica: 20 estimaciones, intervalo del 90 % ----
n_test = 20
verdad = np.zeros(n_test)
estimado = aleatorio.normal(0.0, SIGMA_REAL, n_test)
sigma_declarada = SIGMA_REAL / 3
medio_ancho = norm.ppf(0.95) * sigma_declarada

dentro = np.abs(estimado - verdad) <= medio_ancho
for i in range(n_test):
    color = C.green if dentro[i] else C.red
    ax2.plot([estimado[i] - medio_ancho, estimado[i] + medio_ancho],
             [i, i], color=color, lw=2.4, alpha=0.85)
    ax2.plot(estimado[i], i, "o", color=color, ms=4)
ax2.axvline(0, color=C.ink, lw=1.6)
ax2.text(0.04, n_test - 0.5, "valor real", color=C.ink, fontsize=8.6)
ax2.set_yticks([])
ax2.set_xlabel("error de la estimación (décadas)")
ax2.set_title(f"Intervalos «del 90 %» de alguien con exceso de confianza:\n"
              f"aciertan {dentro.sum()} de {n_test}", fontsize=10)
ax2.grid(axis="y", alpha=0)

print(f"cobertura real del intervalo declarado al 90 %: {dentro.mean():.0%}")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué multiplicar seis números malos puede dar un resultado bueno?

Simula estimaciones con n factores, cada uno con el mismo error logarítmico
independiente, y compara el error total con las dos hipótesis extremas:
que los errores se sumen (peor caso) o que se cancelen en raíz de n.

La figura responde: ¿cuánto crece realmente el error de una estimación al
añadir factores?

Ejecutar:  python fig_cancelacion.py

*(script original: `codigo/fig_cancelacion.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
aleatorio = rng(7)

SIGMA = 0.30            # incertidumbre de cada factor, en décadas (dex)
N_MAX = 12
N_MUESTRAS = 40_000
ns = np.arange(1, N_MAX + 1)

# Cada factor aporta un error log-normal de anchura SIGMA dex
errores = aleatorio.normal(0.0, SIGMA, size=(N_MUESTRAS, N_MAX))
acumulado = np.cumsum(errores, axis=1)
sigma_simulada = acumulado.std(axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 4.0))

# --- Panel 1: crecimiento del error ---------------------------------------
ax1.plot(ns, SIGMA * ns, "--", color=C.red, lw=1.6,
         label="si los errores se sumaran:  $n\\sigma$")
ax1.plot(ns, SIGMA * np.sqrt(ns), "-", color=C.blue, lw=2.0,
         label="si son independientes:  $\\sqrt{n}\\,\\sigma$")
ax1.plot(ns, sigma_simulada, "o", color=C.ink, ms=5,
         label="simulación (40 000 estimaciones)")
ax1.set_xlabel("número de factores $n$")
ax1.set_ylabel("incertidumbre total (décadas)")
ax1.set_title("El error crece como $\\sqrt{n}$, no como $n$")
ax1.legend(loc="upper left")

# Eje derecho: la misma incertidumbre leída como «factor de error»
sec = ax1.secondary_yaxis(
    "right", functions=(lambda d: d, lambda d: d))
sec.set_yticks([np.log10(f) for f in (2, 3, 10, 30, 100, 1000)])
sec.set_yticklabels([f"×{f}" for f in (2, 3, 10, 30, 100, 1000)], fontsize=8)
sec.set_ylabel("factor de error equivalente", fontsize=9)

# --- Panel 2: distribución del resultado para n = 6 -----------------------
n_demo = 6
muestras = acumulado[:, n_demo - 1]
ax2.hist(muestras, bins=90, color=C.blue, alpha=0.55, density=True,
         edgecolor="none", label=f"$n={n_demo}$ factores")
ax2.hist(acumulado[:, 0], bins=90, color=C.grey, alpha=0.45, density=True,
         edgecolor="none", label="$n=1$ factor")

for q, estilo in [(0.05, ":"), (0.95, ":")]:
    ax2.axvline(np.quantile(muestras, q), color=C.red, ls=estilo, lw=1.3)
p05, p95 = np.quantile(muestras, [0.05, 0.95])
ax2.annotate(
    f"el 90 % cae dentro\nde un factor {10**((p95-p05)/2):.0f} arriba o abajo",
    xy=(p95, 0.35), xytext=(p95 + 0.15, 0.95), color=C.red, fontsize=8.6,
    arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
ax2.set_xlabel("error del resultado (décadas)")
ax2.set_ylabel("densidad")
ax2.set_title("Seis factores mediocres, un resultado decente")
ax2.legend(loc="upper left")
ax2.set_xlim(-3, 3)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cabe una tormenta en la misma escala que una bomba y que un mosquito?

Escalera logarítmica de energías: 60 órdenes de magnitud en un solo eje.
La figura responde: ¿dónde cae una tormenta de verano en el mapa de las
energías, y qué tiene al lado?

Ejecutar:  python fig_escala_energias.py

*(script original: `codigo/fig_escala_energias.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# (energía en julios, etiqueta, familia)
ENERGIAS = [
    (1.6e-19, "1 eV · enlace químico débil", "micro"),
    (3.2e-19, "Fotón visible (≈2 eV)", "micro"),
    (3.2e-11, "Fisión de un núcleo de $^{235}$U", "micro"),
    (3.0e-7,  "Mosquito volando", "cotidiano"),
    (1.5e0,   "Manzana cayendo desde 1 m", "cotidiano"),
    (3.6e5,   "Barrita de cereales (≈85 kcal)", "cotidiano"),
    (2.3e6,   "Evaporar 1 L de agua", "cotidiano"),
    (8.6e6,   "Una persona en un día (100 W)", "cotidiano"),
    (4.2e9,   "1 tonelada de TNT", "grande"),
    (5.0e9,   "Un rayo", "grande"),
    (6.3e13,  "Hiroshima (≈15 kt)", "grande"),
    (4.5e15,  "TORMENTA DE VERANO (calor latente)", "destacado"),
    (6.3e16,  "Terremoto de magnitud 8", "grande"),
    (2.1e17,  "Tsar Bomba (50 Mt)", "grande"),
    (5.0e18,  "España: energía primaria en un año", "grande"),
    (5.2e19,  "Huracán, un día (calor latente)", "grande"),
    (1.5e22,  "Luz solar sobre la Tierra en un día", "grande"),
    (3.8e26,  "El Sol durante un segundo", "cosmico"),
    (2.1e29,  "Energía de rotación de la Tierra", "cosmico"),
    (1.0e44,  "Supernova (energía cinética)", "cosmico"),
]

COLORES = {"micro": C.grey, "cotidiano": C.blue, "grande": C.ochre,
           "cosmico": C.purple, "destacado": C.red}

fig, ax = plt.subplots(figsize=(7.4, 8.8))

datos = sorted(ENERGIAS)
y_real = np.array([np.log10(e) for e, _, _ in datos])
lo, hi = y_real.min() - 3, y_real.max() + 3

# --- Colocación de etiquetas sin solape -----------------------------------
# Alternamos columnas y después separamos verticalmente dentro de cada una.
SEPARACION = 2.6          # décadas mínimas entre etiquetas de la misma columna


def separar(ys: np.ndarray, sep: float) -> np.ndarray:
    """Empuja etiquetas hacia arriba hasta respetar la separación mínima."""
    y = ys.astype(float).copy()
    for _ in range(200):
        movido = False
        for i in range(1, len(y)):
            hueco = y[i] - y[i - 1]
            if hueco < sep:
                ajuste = (sep - hueco) / 2
                y[i - 1] -= ajuste
                y[i] += ajuste
                movido = True
        if not movido:
            break
    return y


idx_der = list(range(0, len(datos), 2))
idx_izq = list(range(1, len(datos), 2))
y_etiqueta = np.zeros(len(datos))
y_etiqueta[idx_der] = separar(y_real[idx_der], SEPARACION)
y_etiqueta[idx_izq] = separar(y_real[idx_izq], SEPARACION)

# --- Eje ------------------------------------------------------------------
ax.vlines(0, lo, hi, color=C.ink, lw=1.6, zorder=2)
for d in range(-20, 46, 10):
    if lo < d < hi:
        ax.hlines(d, -0.045, 0.045, color=C.ink, lw=1.1, zorder=3)
        ax.text(-0.062, d, rf"$10^{{{d}}}$", ha="right", va="center",
                fontsize=8, color=C.ink, alpha=0.55)

# --- Puntos y etiquetas ---------------------------------------------------
X_COL = 0.36
for i, (energia, etiqueta, familia) in enumerate(datos):
    y0 = y_real[i]
    y1 = y_etiqueta[i]
    lado = 1 if i in idx_der else -1
    color = COLORES[familia]
    destacado = familia == "destacado"

    # conector en codo: del eje al texto
    ax.plot([0, lado * X_COL * 0.55, lado * X_COL * 0.92],
            [y0, y1, y1], color=color, lw=0.9, alpha=0.7, zorder=2)
    ax.plot(0, y0, "o", color=color, ms=8 if destacado else 5,
            mec=C.paper, mew=0.8, zorder=5)
    ax.text(lado * X_COL, y1, etiqueta,
            ha="left" if lado > 0 else "right", va="center",
            fontsize=9.2 if destacado else 8.4, color=color,
            weight="bold" if destacado else "normal")

# --- La comparación que da título al capítulo -----------------------------
y_h, y_t = np.log10(6.3e13), np.log10(4.5e15)
ax.annotate("", xy=(0.05, y_t), xytext=(0.05, y_h),
            arrowprops=dict(arrowstyle="<->", color=C.red, lw=1.4))
ax.text(0.078, (y_h + y_t) / 2 + 0.35, "1,9 décadas\n≈ factor 70",
        color=C.red, fontsize=8.4, va="center", ha="left", linespacing=1.3,
        bbox=dict(facecolor=C.paper, edgecolor="none", pad=1.5))

ax.set_xlim(-1.05, 1.05)
ax.set_ylim(lo, hi)
ax.axis("off")
ax.set_title("Energía en julios. Cada marca del eje es un factor $10^{10}$",
             fontsize=10, pad=14)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Sirve de algo acotar cuando no sé estimar?

Compara dos estrategias para el mismo problema (número de coches circulando
simultáneamente en España a las 8 de la mañana): estimar de frente, o acotar
por arriba y por abajo y tomar la media geométrica.

La figura responde: ¿cuánto se estrecha la respuesta al usar cotas absurdas
pero seguras?

Ejecutar:  python fig_sandwich.py

*(script original: `codigo/fig_sandwich.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

CASOS = [
    # etiqueta, cota inferior, cota superior, razonamiento
    ("Cota trivial\n(no puede ser)", 1e3, 3.5e7,
     "más de mil, y menos que\nla población de España"),
    ("Cota razonada\n(parque móvil)", 1e5, 2.5e7,
     "no más que los coches\nmatriculados (≈25 M)"),
    ("Cota afinada\n(hora punta)", 1.5e6, 8e6,
     "entre el 5 % y el 30 %\ndel parque, a las 8:00"),
]
VALOR_REAL = 3.5e6      # estimación independiente por horas-coche anuales

fig, ax = plt.subplots(figsize=(8.2, 4.0))

for i, (etiqueta, lo, hi, razon) in enumerate(CASOS):
    y = len(CASOS) - i
    media_geo = np.sqrt(lo * hi)
    factor = np.sqrt(hi / lo)
    ax.plot([lo, hi], [y, y], color=C.blue, lw=7, alpha=0.28,
            solid_capstyle="butt")
    ax.plot([lo, lo], [y - 0.16, y + 0.16], color=C.blue, lw=2)
    ax.plot([hi, hi], [y - 0.16, y + 0.16], color=C.blue, lw=2)
    ax.plot(media_geo, y, "D", color=C.ink, ms=7, zorder=5)
    ax.text(media_geo, y + 0.26, f"media geométrica · factor {factor:.0f}",
            ha="center", fontsize=8.2, color=C.ink)
    ax.text(1.2e2, y, etiqueta, ha="left", va="center", fontsize=8.8,
            color=C.ink)
    ax.text(6e7, y, razon, ha="left", va="center", fontsize=8.0, color=C.grey)

ax.axvline(VALOR_REAL, color=C.red, lw=1.8)
ax.text(VALOR_REAL * 1.15, 0.42, "estimación independiente\n≈ 3–4 millones", color=C.red,
        fontsize=8.6, va="bottom")

ax.set_xscale("log")
ax.set_xlim(1e2, 6e8)
ax.set_ylim(0.3, len(CASOS) + 0.85)
ax.set_yticks([])
ax.set_xlabel("coches circulando simultáneamente en España a las 8:00")
ax.set_title("Acotar y tomar la media geométrica: cada refinamiento divide\n"
             "el factor de error, no la respuesta")
ax.grid(axis="y", alpha=0)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Qué forma tiene la incertidumbre de una estimación de Fermi?

Propaga la incertidumbre de los cuatro factores de la tormenta por Monte
Carlo y dibuja la distribución del resultado. La figura responde: si no sé
los factores con exactitud, ¿qué intervalo puedo defender para la energía?

Ejecutar:  python fig_tormenta_mc.py

*(script original: `codigo/fig_tormenta_mc.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
aleatorio = rng(1945)
N = 200_000

# Cada factor: (valor central, factor de incertidumbre a 1 sigma)
# «factor 2» significa: creo que está entre valor/2 y valor*2 con ~68 % de
# confianza. En décadas, sigma = log10(factor).
FACTORES = {
    "Área $A$ (m$^2$)":            (1.0e8, 2.5),
    "Lluvia $h$ (m)":              (2.0e-2, 2.0),
    "Densidad $\\rho$ (kg/m$^3$)": (1.0e3, 1.02),
    "Calor latente $L$ (J/kg)":    (2.26e6, 1.02),
}

log_total = np.zeros(N)
sigmas = {}
for nombre, (centro, factor) in FACTORES.items():
    sigma = np.log10(factor)
    sigmas[nombre] = sigma
    log_total += np.log10(centro) + aleatorio.normal(0.0, sigma, N)

energia = 10**log_total
p05, p50, p95 = np.percentile(energia, [5, 50, 95])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.1),
                               gridspec_kw={"width_ratios": [1.5, 1]})

# --- Distribución ---------------------------------------------------------
ax1.hist(log_total, bins=140, color=C.blue, alpha=0.6, edgecolor="none")
alto = ax1.get_ylim()[1]
ax1.set_ylim(0, alto * 1.30)
etiquetas = [
    (np.log10(p05), f"P5\n{p05:.0e} J", C.grey, "--", 1.06, "right"),
    (np.log10(p50), f"mediana\n{p50:.0e} J", C.ink, "-", 1.22, "center"),
    (np.log10(p95), f"P95\n{p95:.0e} J", C.grey, "--", 1.06, "left"),
]
for x, etiqueta, color, estilo, altura, ali in etiquetas:
    ax1.axvline(x, color=color, lw=1.4, ls=estilo)
    ax1.text(x, alto * altura, etiqueta, ha=ali, va="bottom",
             fontsize=8.2, color=color, linespacing=1.25)

ax1.axvline(np.log10(6.3e13), color=C.red, lw=1.6)
ax1.annotate("Hiroshima", xy=(np.log10(6.3e13), alto * 0.45),
             xytext=(np.log10(6.3e13) - 1.6, alto * 0.72), color=C.red,
             fontsize=8.6,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))

ax1.set_xlabel("$\\log_{10}$ de la energía liberada (J)")
ax1.set_ylabel("frecuencia")
ax1.set_title("La incertidumbre es log-normal, no normal")
ax1.set_yticks([])

# --- ¿Quién manda en el error? -------------------------------------------
nombres = list(sigmas)
varianzas = np.array([sigmas[n] ** 2 for n in nombres])
contrib = 100 * varianzas / varianzas.sum()
orden = np.argsort(contrib)
colores = [C.red if c > 20 else C.grey for c in contrib[orden]]
ax2.barh([nombres[i] for i in orden], contrib[orden], color=colores, height=0.6)
for i, (idx) in enumerate(orden):
    ax2.text(contrib[idx] + 1.5, i, f"{contrib[idx]:.0f} %",
             va="center", fontsize=8.6, color=C.ink)
ax2.set_xlabel("contribución a la varianza total (%)")
ax2.set_xlim(0, 100)
ax2.set_title("Mejorar $\\rho$ o $L$ no sirve de nada")
ax2.grid(axis="y", alpha=0)

print(f"mediana = {p50:.2e} J   P5 = {p05:.2e} J   P95 = {p95:.2e} J")
print(f"factor entre P5 y P95: {p95/p05:.0f}")
print(f"equivalente en Hiroshimas: {p50/6.3e13:.0f}")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
